# TDA Mapper Implementation

In [ ]:
import pandas as pd
import kmapper as km
import sklearn
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import umap

## Data Processing

In [ ]:
db = pd.read_csv('database.csv')
db = db[db.columns[1:]]
db.rename(columns = {"Cantidad_Entregada": "Lts Consumidos"}, inplace = True)

Make the model year an integer.

In [ ]:
db["model_year"] = db["model_year"].apply(lambda x: int(x))

Add the columns CO2/L.

In [ ]:
db["CO2/Km"] = db["CO2"] / db["Distancia"]

Keep only a row for each VIN with the most recent register.

In [ ]:
db.sort_values(by = ["Año", "Mes"], inplace = True)
db.drop_duplicates(subset = "VIN", keep ="last", inplace = True)
db.reset_index(drop = True, inplace = True)

Dataframe with the models and its producer.

In [ ]:
models = db.drop_duplicates(subset = ["make", "model"]) \
         .sort_values("make")[["make", "model"]] \
         .reset_index(drop = True)

Select the required variables.

In [ ]:
db = db[["CO2/Km", "Rendimiento Real", "VIN", "model_year", "make", "model", "body_class"]]

In [ ]:
db_num = db.copy().drop("VIN", axis = 1).drop("make", axis = 1)

In [ ]:
db["body_class"].unique()

array(['Pickup', 'Sedan', 'SUV', 'Van', 'Truck', 'Truck-Tractor',
       'Chassis Cab', 'Sedan/Saloon'], dtype=object)

In [ ]:
db["body_class"].value_counts()

,count
body_class,
Pickup,196
Truck-Tractor,95
SUV,57
Van,39
Truck,38
Chassis Cab,17
Sedan,1
Sedan/Saloon,1


### Ordinal Encoding Mapper Algorithm Data

In [ ]:
order_body_class = ['Sedan', 'Sedan/Saloon', 'SUV', 'Van', 'Pickup',
                    'Chassis Cab', 'Truck', 'Truck-Tractor']  # From smallest to largest value
order_model = models["model"].values

In [ ]:
# By default value starts from 0, so add 1 to start from 1
db_num["body_class"] = (
    pd.Categorical(db["body_class"],
                   categories = order_body_class,
                   ordered=True).codes + 1
)

In [ ]:
# By default value starts from 0, so add 1 to start from 1
db_num["model"] = (
    pd.Categorical(db["model"],
                   categories = order_model,
                   ordered=True).codes + 1
)

### Colors and Tooltips for Mapper

In [ ]:
# List with values for coloring the nodes
colores = db_num[['CO2/Km', 'model_year', 'Rendimiento Real']].values
colores = np.concatenate((colores,
                          (2025-db["model_year"]).values.reshape(-1, 1)), axis = 1)

In [ ]:
# Tooltips
tooltips = []
for row in db[["CO2/Km",	"Rendimiento Real",	"model_year", "make",	"model",	"body_class"]].values:
  tp = "[CO2/Km: " + f"{row[0]:.2f}" # TON CO2
  tp += ", Km/L: " + str(row[1]) # Real performance
  tp += f", {row[3]}" # Producer
  tp += f" {row[4]}" # Model
  tp += f" {row[2]}" # Year
  tp += ", " + str(row[5]) # Truck type
  tp += "]"
  tooltips.append(tp)

tooltips = np.array(tooltips)

### Filter for Mapper

In [ ]:
# Minmax scaler
scaler = MinMaxScaler(feature_range = (0, 1))
# Filtro UMAP
reducer = umap.UMAP(n_components = 2)

In [ ]:
# Construction of UMAP filter
val_umap = reducer.fit_transform(db_num)
val_umap = scaler.fit_transform(val_umap)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
# CO2 Values
val_co2Km = scaler.fit_transform(db_num['CO2/Km'].values.reshape(-1, 1))

## Mapper Algorithm

In [ ]:
mapper = km.KeplerMapper(verbose = 1)

graph = mapper.map(
    #np.concatenate((val_umap, val_co2Km), axis = 1),
    db_num[["Rendimiento Real", "model_year", "model", "body_class"]],
    db_num,
    clusterer = sklearn.cluster.DBSCAN(metric = "cosine", eps = 0.4, min_samples = 4),
    cover = km.Cover(n_cubes = 5, perc_overlap = 0.3)
)

mapper.visualize(
    graph,
    path_html = "Mapper_CO2_VIN.html",
    title = "Topología",
    color_values = colores,
    color_function_name = ["CO2/Km", "Año del Modelo", "Rendimiento Real", "Antiguedad"],
    custom_tooltips = tooltips,
    include_searchbar = True,
)

KeplerMapper(verbose=1)
Mapping on data shaped (444, 5) using lens shaped (444, 4)

Creating 625 hypercubes.

Created 105 edges and 40 nodes in 0:00:00.482869.
Wrote visualization to: Mapper_CO2_VIN.html


'<!DOCTYPE html>\n<html>\n\n<head>\n  <meta charset="utf-8">\n  <meta name="generator" content="KeplerMapper">\n  <title>Topología de EDENRED | KeplerMapper</title>\n\n  <link rel="icon" type="image/png" href="http://i.imgur.com/axOG6GJ.jpg" />\n\n  <link href=\'https://fonts.googleapis.com/css?family=Roboto+Mono:700,300\' rel=\'stylesheet\' type=\'text/css\'>\n  <style>* {\n  margin: 0;\n  padding: 0;\n}\n\nhtml, body {\n  height: 100%;\n}\n\nbody {\n  font-family: "Roboto Mono", "Helvetica", sans-serif;\n  font-size: 14px;\n}\n\n#logo {\n  width:  85px;\n  height: 85px;\n}\n\n#display {\n  color: #95A5A6;\n  background: #212121;\n}\n\n#header {\n  background: #111111;\n}\n\n#print {\n  color: #000;\n  background: #FFF;\n}\n\nh1 {\n  font-size: 21px;\n  font-weight: 300;\n  font-weight: 300;\n}\n\nh2 {\n  font-size: 18px;\n  padding-bottom: 20px;\n  font-weight: 300;\n}\n\nh3 {\n  font-size: 14px;\n  font-weight: 700;\n  text-transform: uppercase;\n}\n\nh4 {\n  font-size: 13px;\n  fon